In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

from sklearn.metrics import (
   accuracy_score,
   precision_score,
   recall_score,
   f1_score
)


In [4]:
# Load Titanic dataset directly from an online source to prevent FileNotFoundError
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

In [6]:
# Select features using correct lowercase column names from the Seaborn dataset
features = [
   "pclass", "sex", "age", "sibsp",
   "parch", "fare", "embarked"
]

X = df[features]
y = df["survived"]

In [7]:
# Feature types
numeric_features = [
   "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = ["Sex", "Embarked"]


In [8]:
# Preprocessing
preprocessor = ColumnTransformer(
   transformers=[
       ("num", "passthrough", numeric_features),
       ("cat", OneHotEncoder(handle_unknown="ignore"),
        categorical_features)
   ]
)

In [9]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
   X, y, test_size=0.2,
   random_state=42, stratify=y
)


In [10]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
   X, y, test_size=0.2,
   random_state=42, stratify=y
)


In [12]:
# Define the base estimator
base_model = DecisionTreeClassifier(random_state=42)

# Correct feature name casings to match the dataset
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

# Recreate preprocessor with correct lowercase feature lists
preprocessor = ColumnTransformer(
   transformers=[
       ("num", "passthrough", numeric_features),
       ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
   ]
)

# Bagging
bagging = BaggingClassifier(
   estimator=base_model,
   n_estimators=50,
   random_state=42
)

bagging_pipeline = Pipeline([
   ("preprocessor", preprocessor),
   ("classifier", bagging)
])

bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)

In [14]:
from sklearn.impute import SimpleImputer

# Create a robust preprocessor that handles missing values (NaNs)
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
   transformers=[
       ("num", numerical_transformer, numeric_features),
       ("cat", categorical_transformer, categorical_features)
   ]
)

# AdaBoost
boosting = AdaBoostClassifier(
   estimator=DecisionTreeClassifier(
       max_depth=2,
       random_state=42
   ),
   n_estimators=50,
   random_state=42
)

boosting_pipeline = Pipeline([
   ("preprocessor", preprocessor),
   ("classifier", boosting)
])

boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

In [15]:
# Evaluation function
def evaluate_model(name, y_true, y_pred):
   print("\n", name)

   print("Accuracy:",
         round(accuracy_score(y_true, y_pred), 4))

   print("Precision:",
         round(precision_score(y_true, y_pred), 4))

   print("Recall:",
         round(recall_score(y_true, y_pred), 4))

   print("F1-score:",
         round(f1_score(y_true, y_pred), 4))


In [16]:
# Evaluate
evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)
5


 Bagging
Accuracy: 0.8101
Precision: 0.7692
Recall: 0.7246
F1-score: 0.7463

 Boosting (AdaBoost)
Accuracy: 0.7765
Precision: 0.7636
Recall: 0.6087
F1-score: 0.6774


5